In [ ]:
import sqlite3
import pandas as pd


In [ ]:
conn = sqlite3.connect("../data/raw/database.db")

In [ ]:
df = pd.read_sql_query("SELECT * FROM player_stats", conn)

In [ ]:
df.head()


In [ ]:
df.shape

In [ ]:
df["rank_tier"].value_counts().sort_index()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
df.groupby("rank_tier")["positioning_goals_against_while_last_defender"].apply(lambda x: x.isna().sum())

In [ ]:
df.groupby("rank_tier")["positioning_goals_against_while_last_defender"].apply(lambda x: (x == 0).sum())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.groupby("rank_tier")["movement_avg_speed"].mean()

In [ ]:
df.groupby("rank_tier")["boost_bpm"].mean()

In [ ]:
rank_means = df.groupby("rank_tier").mean(numeric_only=True)

rank_means


In [ ]:
rank_means[[
    "core_score",
    "core_shots",
    "core_saves",
    "boost_bpm",
    "movement_avg_speed",
    "demo_inflicted"
]]

In [ ]:
rank_means[["boost_bpm","movement_avg_speed"]].plot(kind="line")

In [ ]:
rank_means["boost_bpm"].plot(kind="line")

In [ ]:
rank_correlations = df.corr(numeric_only=True)["rank_tier"].sort_values(ascending=False)

rank_correlations.head(40)

In [ ]:
rank_correlations.tail(20)

In [ ]:
important_correlations = rank_correlations.drop("rank_tier")

important_correlations.head(10)

In [ ]:
important_correlations.tail(10)

In [ ]:
important_correlations.abs().sort_values(ascending=False).head(20)

In [ ]:
correlation_summary = pd.DataFrame({
    "correlation": important_correlations,
    "strength": important_correlations.abs()
}).sort_values("strength", ascending=False)

correlation_summary.head(20)

In [ ]:
correlation_summary.head(20).plot(kind="barh", y="correlation")

In [ ]:
rank_means["movement_percent_high_air"].plot(kind="line")

In [ ]:
rank_means["positioning_avg_distance_to_mates"].plot(kind="line")

In [ ]:
rank_means["movement_percent_high_air"].plot(kind="line")

In [ ]:
rank_means["movement_percent_slow_speed"].plot(kind="line")


## Early EDA Findings

The dataset contains player-level Rocket League replay stats from Gold 1 through Grand Champion 3. Each rank tier has 102 player rows, except Grand Champion 3, which has 90 rows.

The clearest rank-separating stats are not basic scoreboard stats like goals, assists, or score. Instead, the strongest patterns are related to movement, aerial play, boost usage, and positioning.

The strongest positive correlations with rank are:

- `movement_percent_high_air`
- `movement_time_high_air`
- `movement_avg_speed`
- `movement_percent_supersonic_speed`
- `boost_bcpm`
- `boost_bpm`
- `movement_count_powerslide`
- `positioning_avg_distance_to_mates`

These stats increase as rank increases. This suggests that higher-ranked players play faster, spend more time in the air, use more boost, reach supersonic speed more often, powerslide more often, and maintain more spacing from teammates.

The strongest negative correlations with rank are:

- `movement_percent_slow_speed`
- `movement_percent_ground`
- `movement_time_slow_speed`
- `movement_avg_powerslide_duration`
- `movement_time_ground`
- `positioning_percent_defensive_third`

These stats decrease as rank increases. This suggests that higher-ranked players spend less time moving slowly, less time grounded, and less time stuck in their defensive third.

One interesting finding is that powerslide count increases with rank, but average powerslide duration decreases. This may mean higher-ranked players powerslide more often, but in shorter and more controlled bursts.

Overall, the early EDA suggests that Rocket League rank is separated more by pace, boost activity, aerial involvement, and movement efficiency than by simple scoreboard stats.

In [ ]:
model_df = df.copy()

In [ ]:
model_df = model_df.drop(columns=[
    "replay_id",
    "rank",
    "team",
])

In [ ]:
model_df = model_df.drop(columns=[
    "positioning_goals_against_while_last_defender"
])

In [ ]:
model_df.isna().sum().sum()

In [ ]:
X = model_df.drop(columns=["rank_tier"])
y = model_df["rank_tier"]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_model = RandomForestClassifier(
    random_state=42
)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

In [ ]:
abs(y_test - y_pred).mean()

In [ ]:
(abs(y_test - y_pred) <= 1).mean()

In [ ]:
(abs(y_test - y_pred) <= 2).mean()

## First Model Results

I trained a baseline Random Forest classifier to predict exact `rank_tier` from player-level replay stats.

Results:

- Exact tier accuracy: 23.0%
- Within 1 tier accuracy: 49.0%
- Within 2 tiers accuracy: 65.5%
- Mean absolute tier error: 2.07 tiers

Since the model is predicting 15 possible tiers, exact accuracy is harsh. The more useful result is that the model predicts within 1 tier about half the time and within 2 tiers about two-thirds of the time. This suggests that the replay stats contain real rank signal, but exact division prediction is still difficult.

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

## Random Forest Feature Importance

The Random Forest model mostly relied on the same types of stats that appeared in the correlation analysis. The most important features were related to aerial play, movement, boost usage, powersliding, and teammate spacing.

Top features included `movement_percent_high_air`, `positioning_avg_distance_to_mates`, `movement_time_high_air`, `movement_count_powerslide`, `movement_percent_supersonic_speed`, `boost_bpm`, and `boost_bcpm`.

This supports the earlier EDA finding that rank is separated more by pace, aerial involvement, boost activity, and movement efficiency than by basic scoreboard stats.

One possible issue is that `duration` appeared as an important feature. Since game length is not really a player skill stat, future model versions may need to remove `duration` and rely more on percentage or per-minute features.

In [ ]:
model_df = model_df.drop(columns=["duration"])

In [ ]:
X = model_df.drop(columns=["rank_tier"])
y = model_df["rank_tier"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
rf_model = RandomForestClassifier(random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
abs(y_test - y_pred).mean()

In [ ]:
(abs(y_test - y_pred) <= 1).mean()

In [ ]:
(abs(y_test - y_pred) <= 2).mean()

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

## Model After Dropping Duration

I removed `duration` from the model features because game length was only meant to be a replay-quality filter, not a player skill stat.

After removing `duration`, the model performance was:

- Exact tier accuracy: 21.7%
- Within 1 tier accuracy: 46.1%
- Within 2 tiers accuracy: 67.8%
- Mean absolute tier error: 2.13 tiers

The exact accuracy dropped slightly, but the model became conceptually cleaner because it no longer uses game length as a predictor. Feature importance still focused on aerial play, speed, boost usage, powersliding, and spacing.

In [ ]:
def tier_to_rank_group(tier):
    if tier <= 9:
        return "gold"
    elif tier <= 12:
        return "platinum"
    elif tier <= 15:
        return "diamond"
    elif tier <= 18:
        return "champion"
    else:
        return "grand_champion"

y_group = model_df["rank_tier"].apply(tier_to_rank_group)

In [ ]:
def test_random_forest(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    exact_accuracy = accuracy_score(y_test, y_pred)

    return model, y_test, y_pred, exact_accuracy

In [ ]:
exact_model, exact_y_test, exact_y_pred, exact_accuracy = test_random_forest(X, y)

exact_accuracy

In [ ]:
group_model, group_y_test, group_y_pred, group_accuracy = test_random_forest(X, y_group)

group_accuracy

## Broad Rank Model

I also tested a simpler target where exact tiers were grouped into broad ranks:

- Gold
- Platinum
- Diamond
- Champion
- Grand Champion

The Random Forest model reached 53.3% accuracy on this broader rank prediction task.

This is much stronger than exact-tier accuracy because the model only has to choose between 5 rank groups instead of 15 exact divisions. This suggests the model is better suited for predicting a player's general rank level than their exact division.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["gold", "platinum", "diamond", "champion", "grand_champion"]

cm = confusion_matrix(group_y_test, group_y_pred, labels=labels)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
).plot()

## Broad Rank Confusion Matrix

The confusion matrix shows that the model performs best at the extremes.

Gold and Grand Champion are predicted more accurately than the middle ranks. The model correctly predicted 41 Gold players as Gold and 50 Grand Champion players as Grand Champion.

The model struggles more with Platinum, Diamond, and Champion. These ranks are often confused with nearby ranks, which makes sense because the statistical differences between middle ranks are less extreme.

This suggests that the model understands general skill level, but exact boundaries between neighboring ranks are still noisy.

In [ ]:
import pickle

with open("../model/model.pkl", "wb") as file:
    pickle.dump(group_model, file)

In [ ]:
with open("../model/features.pkl", "wb") as file:
    pickle.dump(list(X.columns), file)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42))
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_group,
    test_size=0.2,
    random_state=42,
    stratify=y_group
)

log_reg_model.fit(X_train, y_train)

log_reg_pred = log_reg_model.predict(X_test)

accuracy_score(y_test, log_reg_pred)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["gold", "platinum", "diamond", "champion", "grand_champion"]

cm = confusion_matrix(y_test, log_reg_pred, labels=labels)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
).plot()

In [ ]:
with open("../model/log_reg_broad_rank_model.pkl", "wb") as file:
    pickle.dump(log_reg_model, file)